## 1.4 PDF Ingestion with LangChain

### Objective

Understand how LangChain loads PDF documents, how PDF pages are converted into
Document objects, how page-level metadata is generated, and how PDF ingestion
differs from TXT ingestion.

### Note

Our PDF is a relatively clean, text-based PDF.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

In [ ]:
# check whether the PDF file exists in it's path or not
from pathlib import Path

pdf_path = Path("../../data/raw/pdf/azure_event_hubs_knowledge.pdf")

print("Current directory:", Path.cwd())
print("PDF:", pdf_path.resolve())
print("Exists:", pdf_path.exists())

In [ ]:
# Load the PDF file
loader = PyPDFLoader(str(pdf_path))
print(type(loader))
documents = loader.load()

In [ ]:
print(type(documents))

In [ ]:
print("Number of Documents:", len(documents))

In [ ]:
# Inspect the first Document
document = documents[0]
print("Document Type:", type(document))
print(document.page_content)

In [ ]:
# Inspect the metadata
print("Document Metadata:", document.metadata)

## Inspect every page

In [ ]:
for index, document in enumerate(documents):

    print("=" * 70)
    print(f"Document {index + 1}")
    print("=" * 70)

    print("Page:", document.metadata.get("page"))
    print("Source:", document.metadata.get("source"))
    print("Characters:", len(document.page_content))
    print()

    print(document.page_content[:500])
    print()

### Create a simple page map

In [ ]:
for document in documents:

    print(
        f"Page {document.metadata.get('page')} "
        f"→ {len(document.page_content)} characters"
    )

### Let's inspect page numbering

In [ ]:
for document in documents:
    print(document.metadata.get("page"))

### Let's build a reusable PDF inspection function

In [ ]:
def inspect_documents(documents):
    print(f"Number of documents: {len(documents)}")
    print()

    for index, document in enumerate(documents):

        print(f"Document {index + 1}")
        print("-" * 60)

        print("Type:", type(document))
        print("Characters:", len(document.page_content))
        print("Metadata:", document.metadata)

        print("\nContent preview:")
        print(document.page_content[:300])

        print()

In [ ]:
# call the function to inspect the documents
inspect_documents(documents)

### Note
A real-world PDF problem. A real enterprise PDFs can look like

            PDF
            │
            ├── Text
            ├── Tables
            ├── Images
            ├── Headers
            ├── Footers
            ├── Columns
            ├── Charts
            └── Scanned pages            
A simple PDF text extractor may struggle with:

            Scanned PDF
                │
                ▼
            No actual text layer
                │
                ▼
            Text extraction fails

That's where OCR enters the picture.

We'll study this later under advanced document ingestion rather than mixing it into our first PDF experiment.

### One more experiment — empty pages

Let's check whether any pages contain no meaningful text:
This is a very simple but useful ingestion-quality check.

Why?

Because in real ingestion pipelines you might encounter:

                PDF
                │
                ├── Page 1 → text
                ├── Page 2 → text
                ├── Page 3 → EMPTY
                ├── Page 4 → text
                └── Page 5 → image only

You don't necessarily want to blindly send all of those into your downstream pipeline.

## Stage 1.4 — Key Takeaways

1. PyPDFLoader is a LangChain document loader for PDF files.
2. A PDF loader converts PDF content into LangChain Document objects.
3. With PyPDFLoader, a text-based PDF is commonly represented as one Document per page.
4. Each Document contains page_content and metadata.
5. PDF metadata can include the source and page number.
6. Page numbers may be zero-based in metadata.
7. PDF ingestion is different from TXT ingestion.
8. Loading a PDF is not the same as chunking the PDF.
9. A PDF page is not the same thing as a RAG chunk.
10. Real-world PDFs can contain tables, images, columns, headers, footers, and scanned pages.
11. Scanned PDFs may require OCR because they may not contain an actual text layer.
12. Document ingestion quality has a direct impact on the downstream RAG pipeline.